In [1]:
!pip -q install -U transformers peft bitsandbytes accelerate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 69.4 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.4/637.4 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 85.8 MB/s eta 0:00:00:00:01


In [2]:
!pip -q install "pandas==2.2.2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 88.3 MB/s eta 0:00:00:00:01:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


In [3]:
import pandas as pd
print(pd.__version__)

2.2.2


In [4]:
import torch, pandas as pd, bitsandbytes as bnb, transformers
print("cuda:", torch.cuda.is_available())
print("pandas:", pd.__version__)
print("bnb:", bnb.__version__)
print("transformers:", transformers.__version__)

cuda: True
pandas: 2.2.2
bnb: 0.49.2
transformers: 5.5.0


In [5]:
import json, random
from datasets import load_dataset

random.seed(42)

must_have = [
    "What is the capital of France?","What is the capital of Japan?","What is gravity?",
    "What is photosynthesis?","How do I make a cup of tea?","How do I boil an egg?",
    "What is the difference between RAM and storage?","What causes rain?",
    "Explain recursion simply.","What is 12 + 15?","What is the chemical formula of water?",
    "Who wrote Romeo and Juliet?","What planet is known as the Red Planet?",
    "What is the largest mammal?","How many days are in a leap year?",
    "What is dopamine?","How does Wi-Fi work?","Tips to study better?",
    "How to write a good resume summary?","What is machine learning?"
]

d = load_dataset("databricks/databricks-dolly-15k", split="train")
safe = {"open_qa","closed_qa","information_extraction","summarization","brainstorming"}

pool = []
for x in d:
    if x["category"] in safe:
        p = (x["instruction"] or "").strip()
        if p and 6 <= len(p.split()) <= 30:
            pool.append(p)

random.shuffle(pool)
prompts = must_have[:]
seen = set(p.lower() for p in prompts)
for p in pool:
    if len(prompts) >= 100:
        break
    if p.lower() not in seen:
        prompts.append(p)
        seen.add(p.lower())

with open("/kaggle/working/eval_100_prompts.jsonl", "w", encoding="utf-8") as f:
    for i, p in enumerate(prompts):
        f.write(json.dumps({"id": i, "prompt": p}, ensure_ascii=False) + "\n")

print("Saved prompts:", len(prompts))

README.md: 0.00B [00:00, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

Saved prompts: 100


In [6]:
!mkdir -p /kaggle/output

In [7]:
!cp /kaggle/working/eval_100_prompts.jsonl /kaggle/output/

In [8]:
!wc -l /kaggle/working/eval_100_prompts.jsonl

100 /kaggle/working/eval_100_prompts.jsonl


In [9]:
import json, torch, glob, os
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

model_id = "Qwen/Qwen2.5-7B-Instruct"

  # find adapter path automatically
cand = glob.glob("/kaggle/input/**/dpo_7b_lora/final/adapter_config.json", recursive=True)
if not cand:
    raise FileNotFoundError("Adapter not found in /kaggle/input")
adapter_path = os.path.dirname(cand[0])
print("Adapter path:", adapter_path)

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

tok = AutoTokenizer.from_pretrained(model_id)
tok.pad_token = tok.eos_token

base = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb,
    device_map="auto",
    dtype=torch.float16
)
model = PeftModel.from_pretrained(base, adapter_path)
model.eval()

def ask_7b(q, max_new_tokens=120):
    prompt = f"Human: {q}\n\nAssistant:"
    x = tok(prompt, return_tensors="pt", truncation=True, max_length=1024).to(model.device)
    with torch.no_grad():
        y = model.generate(
            **x,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tok.eos_token_id
        )
    return tok.decode(y[0][x["input_ids"].shape[1]:], skip_special_tokens=True).strip()

rows = []
with open("/kaggle/working/eval_100_prompts.jsonl", "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        ex = json.loads(line)
        rows.append({"id": ex["id"], "prompt": ex["prompt"], "qwen7b_dpo_response": ask_7b(ex["prompt"])})
        if i % 10 == 0:
            print(f"7B done {i}/100")

with open("/kaggle/working/outputs_7b_dpo.jsonl", "w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print("Saved: /kaggle/working/outputs_7b_dpo.jsonl")

Adapter path: /kaggle/input/datasets/arnavx/mini-instructgpt-7b-dpo-adapter/kaggle/working/dpo_7b_lora/final


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


7B done 10/100
7B done 20/100
7B done 30/100
7B done 40/100
7B done 50/100
7B done 60/100
7B done 70/100
7B done 80/100
7B done 90/100
7B done 100/100
Saved: /kaggle/working/outputs_7b_dpo.jsonl


In [10]:
!cp /kaggle/working/outputs_7b_dpo.jsonl /kaggle/output/

In [11]:
!wc -l /kaggle/working/outputs_7b_dpo.jsonl

100 /kaggle/working/outputs_7b_dpo.jsonl


In [12]:
import gc, torch
for v in ["model","base","tok"]:
    if v in globals():
        del globals()[v]
gc.collect()
torch.cuda.empty_cache()
print("GPU cache cleared")

GPU cache cleared


In [16]:
import json, torch, glob, os
from transformers import AutoTokenizer, AutoModelForCausalLM

  # find GPT-2 PPO folder automatically
cand = glob.glob("/kaggle/input/**/ppo_model_v2/config.json", recursive=True)
if not cand:
    raise FileNotFoundError("ppo_model_v2 not found in /kaggle/input")
gpt2_path = os.path.dirname(cand[0])
print("GPT2 path:", gpt2_path)

tok = AutoTokenizer.from_pretrained(gpt2_path, local_files_only=True)
tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(gpt2_path, local_files_only=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device).eval()

def ask_gpt2(q, max_new_tokens=120):
    prompt = f"Human: {q}\n\nAssistant:"
    x = tok(prompt, return_tensors="pt", truncation=True, max_length=1024).to(device)
    with torch.no_grad():
        y = model.generate(
            **x,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tok.eos_token_id
        )
    return tok.decode(y[0][x["input_ids"].shape[1]:], skip_special_tokens=True).strip()

rows = []
with open("/kaggle/working/eval_100_prompts.jsonl", "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        ex = json.loads(line)
        rows.append({"id": ex["id"], "prompt": ex["prompt"], "gpt2_ppo_v2_response": ask_gpt2(ex["prompt"])})
        if i % 10 == 0:
            print(f"GPT2 done {i}/100")

with open("/kaggle/working/outputs_gpt2_ppo_v2.jsonl", "w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print("Saved: /kaggle/working/outputs_gpt2_ppo_v2.jsonl")

GPT2 path: /kaggle/input/datasets/arnavx/mini-instructgpt-ppo-v2-checkpoint/kaggle/working/ppo_model_v2


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2 done 10/100
GPT2 done 20/100
GPT2 done 30/100
GPT2 done 40/100
GPT2 done 50/100
GPT2 done 60/100
GPT2 done 70/100
GPT2 done 80/100
GPT2 done 90/100
GPT2 done 100/100
Saved: /kaggle/working/outputs_gpt2_ppo_v2.jsonl


In [17]:
!mkdir -p /kaggle/output
!cp /kaggle/working/outputs_gpt2_ppo_v2.jsonl /kaggle/output/
!wc -l /kaggle/working/outputs_gpt2_ppo_v2.jsonl

100 /kaggle/working/outputs_gpt2_ppo_v2.jsonl


In [19]:
import json, pandas as pd

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return pd.DataFrame(rows)

d7 = load_jsonl("/kaggle/working/outputs_7b_dpo.jsonl")
d2 = load_jsonl("/kaggle/working/outputs_gpt2_ppo_v2.jsonl")

cmp = d7.merge(d2, on=["id", "prompt"], how="inner")
print("Merged rows:", len(cmp))

markers = [
    "can you clarify","i'm not sure","i am not sure","could you rephrase",
    "what do you mean","need more context","it depends","not sure what you mean"
]

def is_deflect(t):
    s = (t or "").lower()
    return any(m in s for m in markers)

def is_direct(t):
    s = (t or "").strip().lower()
    if len(s.split()) < 8:
        return False
    if is_deflect(s):
        return False
    if s.endswith("?"):
        return False
    return True

cmp["qwen_deflect"] = cmp["qwen7b_dpo_response"].apply(is_deflect)
cmp["gpt2_deflect"] = cmp["gpt2_ppo_v2_response"].apply(is_deflect)
cmp["qwen_direct"] = cmp["qwen7b_dpo_response"].apply(is_direct)
cmp["gpt2_direct"] = cmp["gpt2_ppo_v2_response"].apply(is_direct)

print("Qwen deflection rate:", round(100 * cmp["qwen_deflect"].mean(), 2), "%")
print("GPT2 deflection rate:", round(100 * cmp["gpt2_deflect"].mean(), 2), "%")
print("Qwen direct-answer rate:", round(100 * cmp["qwen_direct"].mean(), 2), "%")
print("GPT2 direct-answer rate:", round(100 * cmp["gpt2_direct"].mean(), 2), "%")

cmp.to_csv("/kaggle/working/comparison_full_100.csv", index=False)
cmp.head(20).to_csv("/kaggle/working/comparison_sample_20.csv", index=False)
print("Saved CSVs to /kaggle/working")

Merged rows: 100
Qwen deflection rate: 2.0 %
GPT2 deflection rate: 88.0 %
Qwen direct-answer rate: 97.0 %
GPT2 direct-answer rate: 1.0 %
Saved CSVs to /kaggle/working


In [20]:
!cp /kaggle/working/comparison_full_100.csv /kaggle/output/

In [21]:
!cp /kaggle/working/comparison_sample_20.csv /kaggle/output/

In [22]:
!ls -lah /kaggle/output

total 200K
drwxr-xr-x 2 root root 4.0K Apr  6 17:51 .
drwxr-xr-x 6 root root 4.0K Apr  6 17:02 ..
-rw-r--r-- 1 root root  75K Apr  6 17:51 comparison_full_100.csv
-rw-r--r-- 1 root root  16K Apr  6 17:51 comparison_sample_20.csv
-rw-r--r-- 1 root root 7.8K Apr  6 17:02 eval_100_prompts.jsonl
-rw-r--r-- 1 root root  63K Apr  6 17:31 outputs_7b_dpo.jsonl
-rw-r--r-- 1 root root  26K Apr  6 17:46 outputs_gpt2_ppo_v2.jsonl
